<a href="https://colab.research.google.com/github/Le2se0hy/FA_ProAn/blob/main/OLS%26SLR_222324.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.neighbors import BallTree
from collections import OrderedDict

# ============================================================
# 0) 유틸
# ============================================================
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * (np.sin(dlon / 2.0) ** 2)
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return R * c

def rmse_fitlm(res):
    return float(np.sqrt(res.mse_resid))

def fit_model(df, y_col, x_cols):
    X = sm.add_constant(df[x_cols], has_constant="add")
    y = df[y_col]
    return sm.OLS(y, X, missing="drop").fit()

# ============================================================
# 1) 중복 좌표 처리 (유일좌표로 묶어서 WY 계산 안정화)
# ============================================================
def make_unique_sum_count(lat, lon, y):
    df_loc = pd.DataFrame({"y": lat, "x": lon, "Y": y})
    g = df_loc.groupby(["y", "x"], sort=True)["Y"].agg(["sum", "count"]).reset_index()
    g["_uix"] = np.arange(len(g), dtype=int)

    lat_uni = g["y"].to_numpy(dtype=float)
    lon_uni = g["x"].to_numpy(dtype=float)
    sumY_uni = g["sum"].to_numpy(dtype=float)
    count_uni = g["count"].to_numpy(dtype=float)

    idx_map = df_loc[["y", "x"]].merge(
        g[["y", "x", "_uix"]],
        on=["y", "x"],
        how="left",
        sort=False
    )["_uix"].to_numpy(dtype=int)

    return lat_uni, lon_uni, sumY_uni, count_uni, idx_map

# ============================================================
# 2) WY 계산 (1km band, inverse-distance, 중복좌표 반영)
# ============================================================
def compute_WY_unique_counts(lat_uni, lon_uni, sumY_uni, count_uni, distance_band_km=1.0, eps=1e-12):
    lat_r = np.deg2rad(lat_uni.astype(float))
    lon_r = np.deg2rad(lon_uni.astype(float))
    coords = np.column_stack([lat_r, lon_r])

    R = 6371.0
    rad_band = distance_band_km / R
    rad_band_candidate = rad_band * (1.0 + 1e-12)

    tree = BallTree(coords, metric="haversine")
    n = len(lat_uni)
    WY_uni = np.zeros(n, dtype=float)

    for i in range(n):
        idx = tree.query_radius(coords[i:i+1], r=rad_band_candidate, return_distance=False)[0]
        idx = idx[idx != i]

        if idx.size == 0:
            WY_uni[i] = 0.0
            continue

        d_km = haversine_km_vec(lat_r[i], lon_r[i], lat_r[idx], lon_r[idx])
        mask = (d_km <= distance_band_km + eps) & (d_km > 0)
        idx2 = idx[mask]
        d2 = d_km[mask]

        if idx2.size == 0:
            WY_uni[i] = 0.0
            continue

        invd = 1.0 / d2
        num = np.sum(invd * sumY_uni[idx2])
        den = np.sum(invd * count_uni[idx2])
        if den == 0:
            den = 1.0

        WY_uni[i] = num / den

    return WY_uni

# ============================================================
# 3) rho 탐색 (RMSE 최소)
# ============================================================
def search_best_rho(df, y_col, x_cols, WY, rho_grid=None):
    if rho_grid is None:
        rho_grid = np.round(np.arange(-0.99, 0.99 + 1e-12, 0.01), 2)

    df_tmp = df.copy()
    df_tmp["_WY_"] = WY

    best_rho = float(rho_grid[0])
    df_tmp["_Y_SPLAG_"] = df_tmp[y_col] - best_rho * df_tmp["_WY_"]
    res_best = fit_model(df_tmp, "_Y_SPLAG_", x_cols)
    best_rmse = rmse_fitlm(res_best)

    for rho in rho_grid[1:]:
        rho = float(rho)
        df_tmp["_Y_SPLAG_"] = df_tmp[y_col] - rho * df_tmp["_WY_"]
        res = fit_model(df_tmp, "_Y_SPLAG_", x_cols)
        r = rmse_fitlm(res)
        if r < best_rmse:
            best_rho, best_rmse, res_best = rho, float(r), res

    return best_rho, best_rmse, res_best

# ================================
# 4) 유의표시: ‡(1%), †(5%)
# ================================
def mark_sig(res, var, digits=3):
    if var not in res.params.index:
        return "–"
    coef = res.params[var]
    pval = res.pvalues[var]
    s = f"{coef:.{digits}f}"  # 음수(-) 자동 표시

    if pval < 0.01:
        s += "‡"
    elif pval < 0.05:
        s += "†"
    return s

def build_table(results_dict, col_order, row_map, obs, f_round_to_10=False):
    T = pd.DataFrame(index=list(row_map.keys()), columns=col_order)

    for col in col_order:
        res = results_dict[col]

        # 계수 출력
        for rname, vname in row_map.items():
            if vname is None:
                T.loc[rname, col] = ""
                continue
            if vname in ["F", "RMSE", "AdjR2"]:
                continue
            T.loc[rname, col] = mark_sig(res, vname)

        # 하단 통계
        fval, fp = res.fvalue, res.f_pvalue
        if fval is None:
            ftxt = "–"
        else:
            fnum = round(fval, -1) if f_round_to_10 else round(fval)
            ftxt = f"{fnum:,.0f}"
            if fp < 0.01:
                ftxt += "‡"
            elif fp < 0.05:
                ftxt += "†"
            elif fp < 0.10:
                ftxt += "+"

        T.loc["F-statistics", col] = ftxt
        T.loc["RMSE", col] = f"{rmse_fitlm(res):.3f}"
        T.loc["Adjusted $R^2$", col] = f"{res.rsquared_adj:.3f}"

    header = f"Obs.= {obs:,}"
    return header, T

# ============================================================
# 5) (중요) 너 파일 컬럼 -> 표준 컬럼으로 변환 + 파생변수 생성
# ============================================================
def prepare_df(excel_path):
    df = pd.read_excel(excel_path)

    # 너가 준 원본 컬럼들 기반 매핑
    #   Price_Won, Latitude, Longitude, Size_m2, Construction_Year,
    #   Year_Sold, Month_Sold, Dist_Subway, Dist_Green, Dist_Water, Dist_CBD,
    #   Bus_Stop, High_School_Count, num_of_people, heating, parking, max_floor
    rename_map = {
        "Price_Won": "Price",
        "Latitude": "y",
        "Longitude": "x",
        "Size_m2": "Area",
        "Construction_Year": "Year",      # 준공연도(건축연도)
        "Year_Sold": "SaleYear",          # 거래연도(분할 기준)
        "Month_Sold": "Month",

        "Dist_Subway": "Dist. Subway",
        "Dist_Green": "Dist. Green",
        "Dist_Water": "Dist. Water",
        "Dist_CBD": "Dist. CBD",

        "Bus_Stop": "Bus Stop",
        "High_School_Count": "High School Cnt",
        "num_of_people": "PopCount",

        "parking": "Parking",
        "max_floor": "MaxFloor",
    }

    for k, v in rename_map.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k: v})

    # Heating 더미 (도시가스=1)
    if "heating" in df.columns and "Heating" not in df.columns:
        df["Heating"] = (df["heating"].astype(str).str.contains("도시가스", na=False)).astype(int)

    # 계절 더미
    if "Month" in df.columns:
        m = pd.to_numeric(df["Month"], errors="coerce")
        df["Spring"] = m.isin([3, 4, 5]).astype(int)
        df["Fall"]   = m.isin([9, 10, 11]).astype(int)
        df["Winter"] = m.isin([12, 1, 2]).astype(int)
    else:
        df["Spring"] = 0
        df["Fall"] = 0
        df["Winter"] = 0

    return df

# ============================================================
# 6) (핵심) 연도별로: WY 계산 -> rho 탐색 -> OLS / SPLAG 회귀 -> 표 출력
# ============================================================
def run_one_year(df_year, distance_band=1.0, f_round_to_10=True, print_summary=True):
    # 필수 컬럼 체크
    required = ["Price", "y", "x"]
    missing = [c for c in required if c not in df_year.columns]

    # 숫자화
    df = df_year.copy()
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    df["x"] = pd.to_numeric(df["x"], errors="coerce")

    # WY 계산 (연도 subset 내부에서만)
    Y = df["Price"].to_numpy(dtype=float)
    lat = df["y"].to_numpy(dtype=float)
    lon = df["x"].to_numpy(dtype=float)

    lat_uni, lon_uni, sumY_uni, count_uni, idx_map = make_unique_sum_count(lat, lon, Y)
    WY_uni = compute_WY_unique_counts(lat_uni, lon_uni, sumY_uni, count_uni, distance_band_km=distance_band)
    df["IND_spw"] = WY_uni[idx_map]

    # =========================
    # ✅ (1)(2)(3) 없애고 "전체 변수" 1번만
    # =========================
    cand = [
        "Area", "Floor", "Parking", "Heating", "Year",
        "Dist. Subway", "Dist. CBD", "Dist. Green", "Dist. Water",
        "Bus Stop", "High School Cnt", "PopCount", "MaxFloor",
        "Spring", "Fall", "Winter",
    ]
    x_full = [c for c in cand if c in df.columns]

    # rho 탐색 (full 기준)
    rho_best, rmse_net, _ = search_best_rho(df, "Price", x_full, df["IND_spw"].to_numpy())

    # OLS (1번)
    res_ols = {"(Full)": fit_model(df, "Price", x_full)}

    # Spatial-lag (1번)
    df["Y_splag_net"] = df["Price"] - rho_best * df["IND_spw"]
    res_splag = {"(Full)": fit_model(df, "Y_splag_net", x_full)}

    obs = int(df[["Price"] + x_full].dropna().shape[0])

    # 표 행 구성(그대로 사용)
    row_map = OrderedDict([
        ("Property characteristics", None),
        ("Size", "Area"),
        ("Floor", "Floor"),
        ("Parking", "Parking"),
        ("Heating(dummy)", "Heating"),
        ("Year built", "Year"),

        ("Accessibility / environment", None),
        ("Dist. subway", "Dist. Subway"),
        ("Dist. CBD", "Dist. CBD"),
        ("Dist. green", "Dist. Green"),
        ("Dist. water", "Dist. Water"),

        ("Local context", None),
        ("Bus stops", "Bus Stop"),
        ("High school cnt", "High School Cnt"),
        ("Population", "PopCount"),
        ("Max floor", "MaxFloor"),

        ("Seasonality control", None),
        ("Spring", "Spring"),
        ("Fall", "Fall"),
        ("Winter", "Winter"),

        ("F-statistics", "F"),
        ("RMSE", "RMSE"),
        ("Adjusted $R^2$", "AdjR2"),
    ])

    h_ols, t_ols = build_table(res_ols, ["(Full)"], row_map, obs, f_round_to_10=f_round_to_10)
    h_sp,  t_sp  = build_table(res_splag, ["(Full)"], row_map, obs, f_round_to_10=f_round_to_10)

    if print_summary:
        print(f"전체 관측치 = {len(df):,}")
        print(f"중복 제거 유일좌표 수 = {len(lat_uni):,}")
        print("Best rho =", rho_best)
        print("Net RMSE =", rmse_net)
        print("OLS RMSE =", rmse_fitlm(res_ols["(Full)"]))

    print("\nTable (OLS):", h_ols)
    print(t_ols)
    print("\nTable (Spatial lag):", h_sp)
    print(t_sp)

    return {
        "df": df,
        "rho_best": rho_best,
        "rmse_net": rmse_net,
        "tables": {"ols": t_ols, "splag": t_sp},
        "results": {"ols": res_ols, "splag": res_splag},
        "x_full": x_full,
    }

def run_by_year(excel_path, years=(2022, 2023, 2024), distance_band=1.0, print_summary=True, f_round_to_10=True):
    df_all = prepare_df(excel_path)

    if "SaleYear" not in df_all.columns:
        raise ValueError(
            "거래연도 컬럼 Year_Sold가 엑셀에 있어야 하고, prepare_df에서 SaleYear로 바뀝니다.\n"
            f"현재 컬럼: {list(df_all.columns)}"
        )

    out = {}
    for y in years:
        df_y = df_all[pd.to_numeric(df_all["SaleYear"], errors="coerce") == y].copy()
        print("\n" + "="*90)
        print(f"[{y}년] 데이터 개수 N={len(df_y):,}")
        print("="*90)

        out[y] = run_one_year(
            df_y,
            distance_band=distance_band,
            f_round_to_10=f_round_to_10,
            print_summary=print_summary
        )

    return out

# ============================================================
# 7) 실행
# ============================================================
if __name__ == "__main__":
    outs = run_by_year(
        excel_path="!Seoul_Apartment_Cleaned_0.5Percent_Trim_Test.xlsx",
        years=(2022, 2023, 2024),
        distance_band=1.0,
        print_summary=True,
        f_round_to_10=True
    )


[2022년] 데이터 개수 N=8,980
전체 관측치 = 8,980
중복 제거 유일좌표 수 = 2,914
Best rho = 0.64
Net RMSE = 406624667.8466794
OLS RMSE = 511412978.8225772

Table (OLS): Obs.= 8,980
                                      (Full)
Property characteristics                    
Size                           14645309.475‡
Floor                           4461947.878‡
Parking                          210934.131‡
Heating(dummy)               -344626804.021‡
Year built                      3546863.372‡
Accessibility / environment                 
Dist. subway                     -86182.976‡
Dist. CBD                         25668.422‡
Dist. green                       55382.064‡
Dist. water                       36256.162‡
Local context                               
Bus stops                      -7127882.270‡
High school cnt                 1684708.302‡
Population                      -126083.937‡
Max floor                       7576311.981‡
Seasonality control                         
Spring                        

In [17]:
import os
import pandas as pd

def save_yearly_results_to_excel(
    outs: dict,
    out_path: str = "Seoul_Results_ByYear.xlsx",
    index_name: str = "Variable"
):

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    summary_rows = []

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for year, pack in outs.items():
            df = pack["df"]

            # 1) 요약 지표 계산/수집
            no_all = int(len(df))

            # 중복 제거 유일좌표 수
            no_uni = int(df[["y", "x"]].dropna().drop_duplicates().shape[0])

            rho_best = float(pack.get("rho_best", float("nan")))
            net_rmse = float(pack.get("rmse_net", float("nan")))

            # OLS RMSE
            ols_res = pack["results"]["ols"]["(Full)"]
            ols_rmse = float(rmse_fitlm(ols_res))

            # OLS Adj R^2
            ols_adj_r2 = float(ols_res.rsquared_adj)

            # SPLAG Adj R^2
            splag_res = pack["results"]["splag"]["(Full)"]
            splag_adj_r2 = float(splag_res.rsquared_adj)

            summary_rows.append({
                "Year": year,
                "Total_Obs(no_all)": no_all,
                "Unique_Coords(no_uni)": no_uni,
                "Best_rho": rho_best,
                "Net_RMSE": net_rmse,
                "OLS_RMSE": ols_rmse,
                "OLS_AdjR2": ols_adj_r2,
                "SPLAG_AdjR2": splag_adj_r2,
            })

            # 2) 표 저장 (OLS / SPLAG)
            t_ols = pack["tables"]["ols"].copy()
            t_sp  = pack["tables"]["splag"].copy()

            t_ols.index.name = index_name
            t_sp.index.name  = index_name

            # 시트명
            t_ols.to_excel(writer, sheet_name=f"OLS_{year}")
            t_sp.to_excel(writer, sheet_name=f"SPLAG_{year}")

        # 3) Summary 시트 저장
        summary_df = pd.DataFrame(summary_rows).sort_values("Year")
        summary_df.to_excel(writer, sheet_name="Summary", index=False)

    print("Saved:", out_path)


# =========================
# 사용 예시
# =========================
# outs = run_by_year(...)
save_yearly_results_to_excel(
    outs,
    out_path="Seoul_Results_ByYear.xlsx"
)

Saved: Seoul_Results_ByYear.xlsx
